# 01. HELOC Preprocess

HELOC 与信データを読み込み、`ClassificationInCredit.md` に合わせた前処理を実行します。  
特殊値の処理、特殊値フラグの付与、特徴量作成、歪度ベースの `log1p` 変換、列メタ情報の保存までを行います。


## ライブラリ


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option("display.max_columns", 200)


## パスと共通設定


In [2]:
BASE_DIR_CANDIDATES = [
    Path.cwd(),
    Path.cwd() / "blog" / "01.Counterfactual_Analysis",
    Path("/home/kohei/WorkSpace/blog/01.Counterfactual_Analysis"),
]

BASE_DIR = next(
    (path for path in BASE_DIR_CANDIDATES if (path / "data" / "data_heloc.csv").exists()),
    None,
)
if BASE_DIR is None:
    raise FileNotFoundError("data/data_heloc.csv が見つかりません。notebook の実行ディレクトリを確認してください。")

DATA_DIR = BASE_DIR / "data"
RAW_DATA_PATH = DATA_DIR / "data_heloc.csv"
DESCRIPTION_PATH = DATA_DIR / "heloc_feature_descriptions.txt"
TRANSFORMED_OUTPUT_PATH = DATA_DIR / "df_loan_transformed.parquet"
TRAIN_OUTPUT_PATH = DATA_DIR / "train_df.parquet"
TEST_OUTPUT_PATH = DATA_DIR / "test_df.parquet"
FEATURE_METADATA_CSV_PATH = DATA_DIR / "heloc_feature_metadata.csv"
FEATURE_METADATA_JSON_PATH = DATA_DIR / "heloc_preprocess_summary.json"

target_col = "is_at_risk"
id_cols = []
datetime_cols = []
cat_columns = []
ordinal_columns = []
flag_columns = []
drop_columns = []
special_value_unique = {-9, -8, -7}
special_value_meanings = {
    -7: "その項目に対応する事象が起きていない",
    -8: "履歴はある前提だが有効な記録がない",
    -9: "信用情報の記録がない、または調査自体が行われていない",
}


## データロード


In [3]:
df_raw = pd.read_csv(RAW_DATA_PATH)
base_feature_columns = [
    col for col in df_raw.columns
    if col not in {target_col, *id_cols, *datetime_cols, *drop_columns}
]
num_columns = base_feature_columns.copy()

print(f"raw shape: {df_raw.shape}")
print(f"target column: {target_col}")
print(f"base feature count: {len(base_feature_columns)}")
display(df_raw.head())


raw shape: (10459, 24)
target column: is_at_risk
base feature count: 23


,estimate_of_risk,months_since_first_trade,months_since_last_trade,average_duration_of_resolution,number_of_satisfactory_trades,nr_trades_insolvent_for_over_60_days,nr_trades_insolvent_for_over_90_days,percentage_of_legal_trades,months_since_last_illegal_trade,maximum_illegal_trades_over_last_year,maximum_illegal_trades,nr_total_trades,nr_trades_initiated_in_last_year,percentage_of_installment_trades,months_since_last_inquiry_not_recent,nr_inquiries_in_last_6_months,nr_inquiries_in_last_6_months_not_recent,net_fraction_of_revolving_burden,net_fraction_of_installment_burden,nr_revolving_trades_with_balance,nr_installment_trades_with_balance,nr_banks_with_high_ratio,percentage_trades_with_balance,is_at_risk
0,55,144,4,84,20,3,0,83,2,3,5,23,1,43,0,0,0,33,-8,8,1,1,69,1
1,61,58,15,41,2,4,4,100,-7,0,8,7,0,67,0,0,0,0,-8,0,-8,-8,0,1
2,67,66,5,24,9,0,0,100,-7,7,8,9,4,44,0,4,4,53,66,4,2,1,86,1
3,66,169,1,73,28,1,1,93,76,6,6,30,3,57,0,5,4,72,83,6,4,3,91,1
4,81,333,27,132,12,0,0,100,-7,7,8,12,0,25,0,1,1,51,89,3,1,0,80,1


## 列説明とメタ情報


In [4]:
def parse_feature_descriptions(path: Path) -> dict[str, str]:
    description_map = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.startswith("- "):
            continue
        body = line[2:]
        if ": " not in body:
            continue
        key, description = body.split(": ", 1)
        description_map[key.strip()] = description.strip()
    return description_map


feature_description_map = parse_feature_descriptions(DESCRIPTION_PATH)
special_value_columns = [
    col for col in base_feature_columns
    if set(df_raw[col].unique()) & special_value_unique
]

feature_metadata_df = pd.DataFrame(
    {
        "feature": base_feature_columns + [target_col],
        "role": ["base_numeric"] * len(base_feature_columns) + ["target"],
        "dtype": [str(df_raw[col].dtype) for col in base_feature_columns + [target_col]],
        "description": [feature_description_map.get(col, "") for col in base_feature_columns + [target_col]],
        "has_special_values": [col in special_value_columns for col in base_feature_columns] + [False],
        "special_values_present": [
            sorted(set(df_raw[col].unique()) & special_value_unique) if col in base_feature_columns else []
            for col in base_feature_columns + [target_col]
        ],
    }
)

print(f"special-value columns: {len(special_value_columns)}")
display(feature_metadata_df)


special-value columns: 23


,feature,role,dtype,description,has_special_values,special_values_present
0,estimate_of_risk,base_numeric,int64,申込者の信用リスクの推定スコア。大きいほど低リスク寄りと解釈されることが多いです。,True,[-9]
1,months_since_first_trade,base_numeric,int64,最初の信用取引を開始してからの経過月数。信用履歴の長さを表します。,True,"[-9, -8]"
2,months_since_last_trade,base_numeric,int64,直近の信用取引からの経過月数。,True,[-9]
3,average_duration_of_resolution,base_numeric,int64,口座や取引の平均継続期間を表す指標です。,True,[-9]
4,number_of_satisfactory_trades,base_numeric,int64,問題なく維持されている取引口座の件数。,True,[-9]
5,nr_trades_insolvent_for_over_60_days,base_numeric,int64,60日超の延滞や債務不履行があった取引件数。,True,[-9]
6,nr_trades_insolvent_for_over_90_days,base_numeric,int64,90日超の延滞や債務不履行があった取引件数。,True,[-9]
7,percentage_of_legal_trades,base_numeric,int64,重大な問題のない正常取引の割合。,True,[-9]
8,months_since_last_illegal_trade,base_numeric,int64,直近の問題取引からの経過月数。,True,"[-9, -8, -7]"
9,maximum_illegal_trades_over_last_year,base_numeric,int64,過去1年で観測された問題取引数の最大値。,True,[-9]


## Train/Test 分割


In [5]:
train_df, test_df = train_test_split(
    df_raw,
    test_size=0.2,
    random_state=42,
    stratify=df_raw[target_col],
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"train shape before preprocessing: {train_df.shape}")
print(f"test shape before preprocessing: {test_df.shape}")


train shape before preprocessing: (8367, 24)
test shape before preprocessing: (2092, 24)


## 特殊値処理


In [6]:
train_df_transformed = train_df.copy()
test_df_transformed = test_df.copy()

train_all_special_mask = train_df_transformed[base_feature_columns].isin(special_value_unique).all(axis=1)
test_all_special_mask = test_df_transformed[base_feature_columns].isin(special_value_unique).all(axis=1)

train_df_transformed = train_df_transformed.loc[~train_all_special_mask].reset_index(drop=True)
test_df_transformed = test_df_transformed.loc[~test_all_special_mask].reset_index(drop=True)

print(f"dropped all-special rows in train: {int(train_all_special_mask.sum())}")
print(f"dropped all-special rows in test: {int(test_all_special_mask.sum())}")


dropped all-special rows in train: 479
dropped all-special rows in test: 109


In [7]:
for col in base_feature_columns:
    for special_value in sorted(special_value_unique):
        flag_col = f"{col}_is_special_{abs(special_value)}"
        train_df_transformed[flag_col] = (train_df_transformed[col] == special_value).astype(int)
        test_df_transformed[flag_col] = (test_df_transformed[col] == special_value).astype(int)
        if train_df_transformed[flag_col].sum() > 0 or test_df_transformed[flag_col].sum() > 0:
            flag_columns.append(flag_col)

flag_columns = sorted(flag_columns)
print(f"special flag count: {len(flag_columns)}")
print(flag_columns)


special flag count: 12
['estimate_of_risk_is_special_9', 'months_since_first_trade_is_special_8', 'months_since_last_illegal_trade_is_special_7', 'months_since_last_illegal_trade_is_special_8', 'months_since_last_inquiry_not_recent_is_special_7', 'months_since_last_inquiry_not_recent_is_special_8', 'net_fraction_of_installment_burden_is_special_8', 'net_fraction_of_revolving_burden_is_special_8', 'nr_banks_with_high_ratio_is_special_8', 'nr_installment_trades_with_balance_is_special_8', 'nr_revolving_trades_with_balance_is_special_8', 'percentage_trades_with_balance_is_special_8']


In [8]:
train_df_transformed.loc[:, base_feature_columns] = train_df_transformed[base_feature_columns].replace(-7, 0)
test_df_transformed.loc[:, base_feature_columns] = test_df_transformed[base_feature_columns].replace(-7, 0)

for col in base_feature_columns:
    train_df_transformed[col] = train_df_transformed[col].astype(float)
    test_df_transformed[col] = test_df_transformed[col].astype(float)

train_special_mask = train_df_transformed[base_feature_columns].isin({-9, -8})
test_special_mask = test_df_transformed[base_feature_columns].isin({-9, -8})

train_df_transformed.loc[:, base_feature_columns] = train_df_transformed[base_feature_columns].mask(train_special_mask, np.nan)
test_df_transformed.loc[:, base_feature_columns] = test_df_transformed[base_feature_columns].mask(test_special_mask, np.nan)

median_values = train_df_transformed[base_feature_columns].median()
train_df_transformed.loc[:, base_feature_columns] = train_df_transformed[base_feature_columns].fillna(median_values)
test_df_transformed.loc[:, base_feature_columns] = test_df_transformed[base_feature_columns].fillna(median_values)

print(f"remaining missing in train base features: {int(train_df_transformed[base_feature_columns].isnull().sum().sum())}")
print(f"remaining missing in test base features: {int(test_df_transformed[base_feature_columns].isnull().sum().sum())}")


remaining missing in train base features: 0
remaining missing in test base features: 0


## 特徴量作成


In [9]:
def add_credit_risk_features(input_df: pd.DataFrame) -> pd.DataFrame:
    df_feat = input_df.copy()

    total_trades = df_feat["nr_total_trades"].replace(0, np.nan)
    revolving_balance_trades = df_feat["nr_revolving_trades_with_balance"].clip(lower=0)
    installment_balance_trades = df_feat["nr_installment_trades_with_balance"].clip(lower=0)
    delinquency_60 = df_feat["nr_trades_insolvent_for_over_60_days"].clip(lower=0)
    delinquency_90 = df_feat["nr_trades_insolvent_for_over_90_days"].clip(lower=0)
    inquiries_6m = df_feat["nr_inquiries_in_last_6_months"].clip(lower=0)

    df_feat["credit_history_length"] = df_feat["months_since_first_trade"].clip(lower=0)
    df_feat["recent_activity_gap"] = (
        df_feat["months_since_first_trade"].clip(lower=0)
        - df_feat["months_since_last_trade"].clip(lower=0)
    ).clip(lower=0)
    df_feat["delinquency_trade_sum"] = delinquency_60 + delinquency_90
    df_feat["delinquency_trade_ratio"] = (df_feat["delinquency_trade_sum"] / total_trades).fillna(0)
    df_feat["satisfactory_trade_ratio"] = (
        df_feat["number_of_satisfactory_trades"].clip(lower=0) / total_trades
    ).fillna(0)
    df_feat["balance_trade_ratio"] = (
        (revolving_balance_trades + installment_balance_trades) / total_trades
    ).fillna(0)
    df_feat["revolving_installment_burden_gap"] = (
        df_feat["net_fraction_of_revolving_burden"] - df_feat["net_fraction_of_installment_burden"]
    )
    df_feat["inquiry_pressure"] = (inquiries_6m / total_trades).fillna(0)
    df_feat["recent_inquiry_share"] = (
        df_feat["nr_inquiries_in_last_6_months_not_recent"].clip(lower=0)
        / inquiries_6m.replace(0, np.nan)
    ).fillna(0)
    df_feat["illegal_trade_gap"] = (
        df_feat["maximum_illegal_trades"] - df_feat["maximum_illegal_trades_over_last_year"]
    )
    df_feat["high_ratio_bank_share"] = (
        df_feat["nr_banks_with_high_ratio"].clip(lower=0) / total_trades
    ).fillna(0)

    return df_feat


derived_feature_descriptions = {
    "credit_history_length": "初回取引からの経過月数。",
    "recent_activity_gap": "初回取引からの経過月数と直近取引からの経過月数の差。最近どれだけ取引活動があったかを表す指標。",
    "delinquency_trade_sum": "60日超延滞件数と90日超延滞件数の合計。",
    "delinquency_trade_ratio": "重い延滞件数合計を総取引件数で割った比率。",
    "satisfactory_trade_ratio": "良好に維持された取引件数を総取引件数で割った比率。",
    "balance_trade_ratio": "残高ありのリボと分割払い取引件数の合計を総取引件数で割った比率。",
    "revolving_installment_burden_gap": "リボルビング型負担率と分割払い型負担率の差。",
    "inquiry_pressure": "過去6か月の照会件数を総取引件数で割った比率。",
    "recent_inquiry_share": "過去6か月照会のうち、ごく最近を除いた照会が占める比率。",
    "illegal_trade_gap": "全期間の問題取引最大件数と過去1年の最大件数の差。",
    "high_ratio_bank_share": "高い利用率を示す銀行系口座数を総取引件数で割った比率。",
}


In [10]:
train_df_transformed = add_credit_risk_features(train_df_transformed)
test_df_transformed = add_credit_risk_features(test_df_transformed)

derived_num_columns = sorted(derived_feature_descriptions.keys())
num_columns = sorted(set(num_columns + derived_num_columns))

print(f"derived feature count: {len(derived_num_columns)}")
print(derived_num_columns)


derived feature count: 11
['balance_trade_ratio', 'credit_history_length', 'delinquency_trade_ratio', 'delinquency_trade_sum', 'high_ratio_bank_share', 'illegal_trade_gap', 'inquiry_pressure', 'recent_activity_gap', 'recent_inquiry_share', 'revolving_installment_burden_gap', 'satisfactory_trade_ratio']


## 歪度確認と `log1p` 変換


In [11]:
skew_threshold = 1.0
skew_base_columns = [col for col in num_columns if col in train_df_transformed.columns and col != target_col]

skew_summary = (
    train_df_transformed[skew_base_columns]
    .agg(["skew", "min", "max"])
    .T.rename(columns={"skew": "skewness", "min": "min_value", "max": "max_value"})
    .sort_values("skewness", key=lambda s: s.abs(), ascending=False)
)
skew_summary["abs_skewness"] = skew_summary["skewness"].abs()
skew_summary["recommended_transform"] = np.select(
    [
        (skew_summary["skewness"] >= skew_threshold) & (skew_summary["min_value"] >= 0),
        skew_summary["abs_skewness"] >= skew_threshold,
    ],
    ["log1p", "consider_other_transform"],
    default="keep_as_is",
)

log_candidate_cols = skew_summary.query(
    "skewness >= @skew_threshold and min_value >= 0"
).index.tolist()

display(skew_summary.head(30))
print("log1p candidates:", log_candidate_cols)


,skewness,min_value,max_value,abs_skewness,recommended_transform
inquiry_pressure,20.275319,0.0,15.333333,20.275319,log1p
satisfactory_trade_ratio,12.106138,0.0,29.000000,12.106138,log1p
balance_trade_ratio,10.761247,0.0,16.000000,10.761247,log1p
delinquency_trade_ratio,9.726804,0.0,6.000000,9.726804,log1p
months_since_last_trade,7.494264,0.0,383.000000,7.494264,log1p
nr_inquiries_in_last_6_months_not_recent,7.065657,0.0,66.000000,7.065657,log1p
nr_inquiries_in_last_6_months,6.781563,0.0,66.000000,6.781563,log1p
high_ratio_bank_share,6.427253,0.0,2.000000,6.427253,log1p
nr_trades_insolvent_for_over_90_days,5.592335,0.0,19.000000,5.592335,log1p
delinquency_trade_sum,4.930736,0.0,38.000000,4.930736,log1p


log1p candidates: ['inquiry_pressure', 'satisfactory_trade_ratio', 'balance_trade_ratio', 'delinquency_trade_ratio', 'months_since_last_trade', 'nr_inquiries_in_last_6_months_not_recent', 'nr_inquiries_in_last_6_months', 'high_ratio_bank_share', 'nr_trades_insolvent_for_over_90_days', 'delinquency_trade_sum', 'nr_trades_insolvent_for_over_60_days', 'months_since_last_inquiry_not_recent', 'nr_banks_with_high_ratio', 'nr_installment_trades_with_balance', 'months_since_last_illegal_trade', 'nr_revolving_trades_with_balance', 'nr_trades_initiated_in_last_year']


In [12]:
for transformed_df in [train_df_transformed, test_df_transformed]:
    for col in log_candidate_cols:
        transformed_df[f"{col}_log1p"] = np.log1p(transformed_df[col].clip(lower=0))

log_output_columns = [f"{col}_log1p" for col in log_candidate_cols]
num_columns = sorted(set(num_columns + log_output_columns) - set(log_candidate_cols))
drop_columns = sorted(set(drop_columns + log_candidate_cols))

print(f"log feature count: {len(log_output_columns)}")
print(log_output_columns)


log feature count: 17
['inquiry_pressure_log1p', 'satisfactory_trade_ratio_log1p', 'balance_trade_ratio_log1p', 'delinquency_trade_ratio_log1p', 'months_since_last_trade_log1p', 'nr_inquiries_in_last_6_months_not_recent_log1p', 'nr_inquiries_in_last_6_months_log1p', 'high_ratio_bank_share_log1p', 'nr_trades_insolvent_for_over_90_days_log1p', 'delinquency_trade_sum_log1p', 'nr_trades_insolvent_for_over_60_days_log1p', 'months_since_last_inquiry_not_recent_log1p', 'nr_banks_with_high_ratio_log1p', 'nr_installment_trades_with_balance_log1p', 'months_since_last_illegal_trade_log1p', 'nr_revolving_trades_with_balance_log1p', 'nr_trades_initiated_in_last_year_log1p']


## 最終特徴量と列メタ情報の整備


In [13]:
model_feature_cols = sorted(
    set(num_columns + flag_columns + ordinal_columns + cat_columns) - set(drop_columns)
)

for col in flag_columns:
    base_col, special_suffix = col.rsplit("_is_special_", 1)
    special_value = -int(special_suffix)
    feature_description_map[col] = (
        f"{base_col} が特殊値 {special_value} を持っていたことを表すフラグ。"
        f"意味: {special_value_meanings[special_value]}。"
    )

for col, description in derived_feature_descriptions.items():
    feature_description_map[col] = description

for col in log_output_columns:
    original_col = col.removesuffix("_log1p")
    feature_description_map[col] = f"{original_col} に対して log1p 変換を適用した列。"

feature_role_map = {col: "base_numeric" for col in base_feature_columns}
feature_role_map.update({col: "derived_numeric" for col in derived_num_columns})
feature_role_map.update({col: "special_value_flag" for col in flag_columns})
feature_role_map.update({col: "log_numeric" for col in log_output_columns})
feature_role_map[target_col] = "target"

ordered_metadata_columns = model_feature_cols + [target_col]
feature_metadata_output_df = pd.DataFrame(
    {
        "feature": ordered_metadata_columns,
        "role": [feature_role_map.get(col, "unknown") for col in ordered_metadata_columns],
        "dtype": [str(train_df_transformed[col].dtype) for col in ordered_metadata_columns],
        "description": [feature_description_map.get(col, "") for col in ordered_metadata_columns],
        "in_model_features": [col in model_feature_cols for col in ordered_metadata_columns],
    }
)

print(f"final feature count: {len(model_feature_cols)}")
display(feature_metadata_output_df.head(40))


final feature count: 46


,feature,role,dtype,description,in_model_features
0,average_duration_of_resolution,base_numeric,float64,口座や取引の平均継続期間を表す指標です。,True
1,balance_trade_ratio_log1p,log_numeric,float64,balance_trade_ratio に対して log1p 変換を適用した列。,True
2,credit_history_length,derived_numeric,float64,初回取引からの経過月数。,True
3,delinquency_trade_ratio_log1p,log_numeric,float64,delinquency_trade_ratio に対して log1p 変換を適用した列。,True
4,delinquency_trade_sum_log1p,log_numeric,float64,delinquency_trade_sum に対して log1p 変換を適用した列。,True
5,estimate_of_risk,base_numeric,float64,申込者の信用リスクの推定スコア。大きいほど低リスク寄りと解釈されることが多いです。,True
6,estimate_of_risk_is_special_9,special_value_flag,int64,estimate_of_risk が特殊値 -9 を持っていたことを表すフラグ。意味: 信用...,True
7,high_ratio_bank_share_log1p,log_numeric,float64,high_ratio_bank_share に対して log1p 変換を適用した列。,True
8,illegal_trade_gap,derived_numeric,float64,全期間の問題取引最大件数と過去1年の最大件数の差。,True
9,inquiry_pressure_log1p,log_numeric,float64,inquiry_pressure に対して log1p 変換を適用した列。,True


## 出力用データ作成


In [14]:
train_output_df = train_df_transformed[model_feature_cols + [target_col]].copy()
test_output_df = test_df_transformed[model_feature_cols + [target_col]].copy()
transformed_output_df = pd.concat([train_output_df, test_output_df], axis=0, ignore_index=True)

print(f"train shape after preprocessing: {train_output_df.shape}")
print(f"test shape after preprocessing: {test_output_df.shape}")
print(f"remaining missing values in train: {int(train_output_df.isnull().sum().sum())}")
print(f"remaining missing values in test: {int(test_output_df.isnull().sum().sum())}")
display(train_output_df.head())


train shape after preprocessing: (7888, 47)
test shape after preprocessing: (1983, 47)
remaining missing values in train: 0
remaining missing values in test: 0


,average_duration_of_resolution,balance_trade_ratio_log1p,credit_history_length,delinquency_trade_ratio_log1p,delinquency_trade_sum_log1p,estimate_of_risk,estimate_of_risk_is_special_9,high_ratio_bank_share_log1p,illegal_trade_gap,inquiry_pressure_log1p,maximum_illegal_trades,maximum_illegal_trades_over_last_year,months_since_first_trade,months_since_first_trade_is_special_8,months_since_last_illegal_trade_is_special_7,months_since_last_illegal_trade_is_special_8,months_since_last_illegal_trade_log1p,months_since_last_inquiry_not_recent_is_special_7,months_since_last_inquiry_not_recent_is_special_8,months_since_last_inquiry_not_recent_log1p,months_since_last_trade_log1p,net_fraction_of_installment_burden,net_fraction_of_installment_burden_is_special_8,net_fraction_of_revolving_burden,net_fraction_of_revolving_burden_is_special_8,nr_banks_with_high_ratio_is_special_8,nr_banks_with_high_ratio_log1p,nr_inquiries_in_last_6_months_log1p,nr_inquiries_in_last_6_months_not_recent_log1p,nr_installment_trades_with_balance_is_special_8,nr_installment_trades_with_balance_log1p,nr_revolving_trades_with_balance_is_special_8,nr_revolving_trades_with_balance_log1p,nr_total_trades,nr_trades_initiated_in_last_year_log1p,nr_trades_insolvent_for_over_60_days_log1p,nr_trades_insolvent_for_over_90_days_log1p,number_of_satisfactory_trades,percentage_of_installment_trades,percentage_of_legal_trades,percentage_trades_with_balance,percentage_trades_with_balance_is_special_8,recent_activity_gap,recent_inquiry_share,revolving_installment_burden_gap,satisfactory_trade_ratio_log1p,is_at_risk
0,63.0,0.374693,136.0,0.0,0.0,80.0,0,0.000000,0.0,0.087011,6.0,6.0,136.0,0,0,0,3.091042,1,0,0.000000,2.772589,54.0,0,19.0,0,0,0.000000,0.693147,0.693147,0,1.386294,0,1.098612,11.0,0.0,0.0,0.0,11.0,36.0,91.0,71.0,0,121.0,1.0,-35.0,0.693147,0
1,130.0,0.251314,342.0,0.0,0.0,91.0,0,0.000000,1.0,0.000000,8.0,7.0,342.0,0,1,0,0.000000,0,1,0.000000,3.761200,74.0,1,3.0,0,0,0.000000,0.000000,0.000000,0,1.098612,0,1.098612,14.0,0.0,0.0,0.0,14.0,36.0,100.0,50.0,0,300.0,0.0,-71.0,0.693147,0
2,88.0,0.361013,193.0,0.0,0.0,56.0,0,0.160343,2.0,0.042560,6.0,4.0,193.0,0,0,0,1.386294,0,0,0.000000,3.850148,37.0,0,64.0,0,0,1.609438,0.693147,0.693147,0,1.098612,0,2.197225,23.0,0.0,0.0,0.0,22.0,22.0,78.0,83.0,0,147.0,1.0,27.0,0.671168,1
3,76.0,0.356675,250.0,0.0,0.0,63.0,0,0.000000,2.0,0.133531,6.0,4.0,250.0,0,0,0,0.693147,0,0,0.000000,3.332205,11.0,0,36.0,0,0,0.000000,1.098612,1.098612,0,0.693147,0,1.791759,14.0,0.0,0.0,0.0,12.0,17.0,75.0,75.0,0,223.0,1.0,25.0,0.619039,1
4,60.0,0.295464,255.0,0.0,0.0,72.0,0,0.247836,1.0,0.000000,8.0,7.0,255.0,0,1,0,0.000000,0,0,2.639057,2.639057,14.0,0,85.0,0,0,2.302585,0.000000,0.000000,0,1.098612,0,2.302585,32.0,0.0,0.0,0.0,31.0,22.0,100.0,79.0,0,242.0,0.0,71.0,0.677399,1


## 保存


In [15]:
transformed_output_df.to_parquet(TRANSFORMED_OUTPUT_PATH, index=False)
train_output_df.to_parquet(TRAIN_OUTPUT_PATH, index=False)
test_output_df.to_parquet(TEST_OUTPUT_PATH, index=False)
feature_metadata_output_df.to_csv(FEATURE_METADATA_CSV_PATH, index=False)

preprocess_summary = {
    "raw_shape": list(df_raw.shape),
    "train_shape_before_preprocess": list(train_df.shape),
    "test_shape_before_preprocess": list(test_df.shape),
    "train_shape_after_preprocess": list(train_output_df.shape),
    "test_shape_after_preprocess": list(test_output_df.shape),
    "target_col": target_col,
    "base_feature_columns": base_feature_columns,
    "model_feature_cols": model_feature_cols,
    "num_columns": num_columns,
    "flag_columns": flag_columns,
    "cat_columns": cat_columns,
    "ordinal_columns": ordinal_columns,
    "drop_columns": drop_columns,
    "special_value_unique": sorted(special_value_unique),
    "special_value_columns": special_value_columns,
    "special_rows_dropped_train": int(train_all_special_mask.sum()),
    "special_rows_dropped_test": int(test_all_special_mask.sum()),
    "log_candidate_cols": log_candidate_cols,
    "log_output_columns": log_output_columns,
    "median_values": {k: float(v) for k, v in median_values.items()},
}

FEATURE_METADATA_JSON_PATH.write_text(
    json.dumps(preprocess_summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"saved transformed full data: {TRANSFORMED_OUTPUT_PATH} {transformed_output_df.shape}")
print(f"saved train data: {TRAIN_OUTPUT_PATH} {train_output_df.shape}")
print(f"saved test data: {TEST_OUTPUT_PATH} {test_output_df.shape}")
print(f"saved feature metadata csv: {FEATURE_METADATA_CSV_PATH}")
print(f"saved preprocess summary json: {FEATURE_METADATA_JSON_PATH}")


saved transformed full data: /home/kohei/WorkSpace/blog/01.Counterfactual_Analysis/data/df_loan_transformed.parquet (9871, 47)
saved train data: /home/kohei/WorkSpace/blog/01.Counterfactual_Analysis/data/train_df.parquet (7888, 47)
saved test data: /home/kohei/WorkSpace/blog/01.Counterfactual_Analysis/data/test_df.parquet (1983, 47)
saved feature metadata csv: /home/kohei/WorkSpace/blog/01.Counterfactual_Analysis/data/heloc_feature_metadata.csv
saved preprocess summary json: /home/kohei/WorkSpace/blog/01.Counterfactual_Analysis/data/heloc_preprocess_summary.json
